In [1]:
import os
import geopandas as gpd
import pandas as pd

In [2]:
# ==================== 1. 参数与路径配置 ====================
city_boundary_gpkg = "/mnt/j/macro_analysis_2026/output260519_city_selection/research_area_filter1.gpkg" 
japan_osm_gpkg = "/mnt/j/macro_analysis_2026/dataset/roaddensity/japan_osm.gpkg"

output_folder = "/mnt/j/macro_analysis_2026/output260519_city_selection"
output_csv = os.path.join(output_folder, "density_results_JPN_precise.csv")

# 你精心筛选的 17 个核心 highway 标签
target_highways = [
    "living_street", "motorway", "motorway_link", "path", "primary", 
    "residential", "road", "secondary", "secondary_link", "service", 
    "services", "tertiary", "tertiary_link", "track", "trunk", 
    "trunk_link", "unclassified"
]

# 核心优化：将标签拼接为标准的 SQL WHERE 子句，供数据库底层过滤
# 形式如: "highway IN ('living_street', 'motorway', ...)"
sql_where_clause = f"highway IN ({', '.join([f"'{h}'" for h in target_highways])})"

In [3]:
# ==================== 2. 读取城市边界并筛选 JPN ====================
print("正在读取城市边界要素...")
gdf_all_cities = gpd.read_file(city_boundary_gpkg)

if gdf_all_cities.crs is None:
    gdf_all_cities.set_crs("EPSG:3395", inplace=True)

gdf_jpn_cities = gdf_all_cities[gdf_all_cities['XC_ISO_LST'] == 'JPN'].copy()
total_cities = len(gdf_jpn_cities)
print(f"-> 成功提取日本城市共计: {total_cities} 个")

# 将城市边界转为 4326，用于动态计算每个城市的 WGS84 外接矩形（BBOX）
gdf_jpn_cities_4326 = gdf_jpn_cities.to_crs("EPSG:4326")

正在读取城市边界要素...
-> 成功提取日本城市共计: 109 个


In [5]:
# ==================== 3. 逐个城市：边读边过滤边释放内存 ====================
print(f"\n[核心双重优化开启] 属性过滤 + BBOX 局部分块模式...")
print(f"-> 底层过滤条件: {sql_where_clause}")

results = []
id_col = 'fid_'

for idx, row in enumerate(gdf_jpn_cities.itertuples(), 1):
    city_id = getattr(row, id_col)
    
    # 获取空间几何
    geom_3395 = getattr(row, 'geometry')
    geom_4326 = gdf_jpn_cities_4326.loc[gdf_jpn_cities_4326[id_col] == city_id, 'geometry'].values[0]
    
    # 计算当前城市的外接矩形 (minx, miny, maxx, maxy)
    city_bbox = geom_4326.bounds  
    
    print(f"[{idx}/{total_cities}] 正在流式读取并分析城市 fid_: {city_id} ...")
    
    try:
        # 【双重减负】：同时限制空间方框(bbox) 与 属性白名单(where)
        gdf_local_roads = gpd.read_file(
            japan_osm_gpkg, 
            layer="lines", 
            bbox=city_bbox,
            where=sql_where_clause
        )
        
        if gdf_local_roads.empty:
            total_length_m = 0.0
        else:
            # 投影转换到米制坐标
            gdf_local_roads_3395 = gdf_local_roads.to_crs("EPSG:3395")
            
            # 【已修正 Bug】：用不规则的城市实际边界进行高精度精确裁剪 (clip)
            gdf_clipped = gpd.clip(gdf_local_roads_3395, geom_3395)
            
            total_length_m = gdf_clipped.geometry.length.sum() if not gdf_clipped.empty else 0.0
            
        # 密度的标准几何计算
        city_area_sqm = geom_3395.area
        total_length_km = total_length_m / 1000
        city_area_sqkm = city_area_sqm / 1000000
        road_density = total_length_km / city_area_sqkm if city_area_sqkm > 0 else 0
        
        results.append({
            "fid_": city_id, 
            "XC_ISO_LST": "JPN",
            "city_area_sqkm": city_area_sqkm, 
            "road_length_km": total_length_km,
            "road_density_km_per_sqkm": road_density, 
            "status": "Success"
        })
        
    except Exception as e:
        print(f"  ⚠ 城市 fid_ {city_id} 计算遇到问题: {e}")
        results.append({
            "fid_": city_id, "XC_ISO_LST": "JPN", "city_area_sqkm": None, 
            "road_length_km": None, "road_density_km_per_sqkm": None, "status": f"Failed: {e}"
        })


[核心双重优化开启] 属性过滤 + BBOX 局部分块模式...
-> 底层过滤条件: highway IN ('living_street', 'motorway', 'motorway_link', 'path', 'primary', 'residential', 'road', 'secondary', 'secondary_link', 'service', 'services', 'tertiary', 'tertiary_link', 'track', 'trunk', 'trunk_link', 'unclassified')
[1/109] 正在流式读取并分析城市 fid_: 2165 ...
[2/109] 正在流式读取并分析城市 fid_: 2166 ...
[3/109] 正在流式读取并分析城市 fid_: 2167 ...
[4/109] 正在流式读取并分析城市 fid_: 2168 ...
[5/109] 正在流式读取并分析城市 fid_: 2169 ...
[6/109] 正在流式读取并分析城市 fid_: 2170 ...
[7/109] 正在流式读取并分析城市 fid_: 2171 ...
[8/109] 正在流式读取并分析城市 fid_: 2172 ...
[9/109] 正在流式读取并分析城市 fid_: 2173 ...
[10/109] 正在流式读取并分析城市 fid_: 2174 ...
[11/109] 正在流式读取并分析城市 fid_: 2175 ...
[12/109] 正在流式读取并分析城市 fid_: 2176 ...
[13/109] 正在流式读取并分析城市 fid_: 2177 ...
[14/109] 正在流式读取并分析城市 fid_: 2178 ...
[15/109] 正在流式读取并分析城市 fid_: 2179 ...
[16/109] 正在流式读取并分析城市 fid_: 2180 ...
[17/109] 正在流式读取并分析城市 fid_: 2181 ...
[18/109] 正在流式读取并分析城市 fid_: 2182 ...
[19/109] 正在流式读取并分析城市 fid_: 2183 ...
[20/109] 正在流式读取并分析城市 fid_: 2184 ...
[21/109] 正在流式

In [6]:
# ==================== 4. 保存结果 ====================
df_result = pd.DataFrame(results)
df_result.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"\n🎉 过滤版城市密度计算圆满完成！结果已妥善保存至:\n{output_csv}")


🎉 过滤版城市密度计算圆满完成！结果已妥善保存至:
/mnt/j/macro_analysis_2026/output260519_city_selection/density_results_JPN_precise.csv
